# 전체 데이터 최종 학습 실행 기록

학습은 로컬 Mac GPU에서 별도 Python 프로세스로 실행했습니다. 이 노트북은 해당 실행의 완료를 기다린 뒤 최종 가중치·파일별 참여 횟수·제출 ZIP을 실제 코드로 검증하고 기록합니다. Colab에서 실행한 노트북은 아닙니다. 다시 실행하려면 이 저장소와 `_workspace/full_refit_20260913_fixed`의 원본 실행 결과가 필요합니다.

실제 학습 명령:
```sh
USE_TF=0 .venv/bin/python -u run_full_refit.py --assets _workspace/v2_models/ast --data v2_validation --output _workspace/full_refit_20260913_fixed --device mps --seconds 6600
```

In [ ]:
from pathlib import Path
import time, json
RUN = Path('_workspace/full_refit_20260913_fixed')
DEADLINE = time.time() + 1800
while not (RUN/'finished.json').exists():
    if time.time() > DEADLINE:
        raise TimeoutError('Final refit did not complete; do not mark it ready')
    time.sleep(15)
print(json.loads((RUN/'finished.json').read_text()))

## 최종 checkpoint와 전체 데이터 사용 검증

동결된 가중치의 일치, 학습 대상의 변경, epoch17 checkpoint와 export의 일치, 모든 파일의 학습 횟수를 검사합니다. ZIP 오프라인 재실행 검증은 학습 프로세스가 `finished.json`을 저장하기 전에 수행했습니다.

In [ ]:
from pathlib import Path
import csv,json,shutil,hashlib,zipfile
import torch
from safetensors import safe_open

root=Path.cwd()
run=root/'_workspace/full_refit_20260913_fixed'
out=root/'results/full_refit_20260913'
submission=root/'submissions/AST_full1029_epoch17'
finished=json.loads((run/'finished.json').read_text())
assert finished['status']=='ready_not_submitted'
assert finished['full_training_files']==1029
submission.mkdir(exist_ok=True,parents=True)
for name in ('finished.json','protocol.json','verification.json','full_training_inventory.json'):
 shutil.copy2(run/name,out/name)
for stage in ('warmup','finetune'):
 shutil.copy2(run/stage/'report.json',out/(stage+'_report.json'))
shutil.copy2('/tmp/dacon_full_refit_fixed.log',out/'training.log')

frozen,changed,unchanged=0,[],[]
head_changes=[]
warmup_state=torch.load(run/'warmup/last.pt',weights_only=False,map_location='cpu')['model']
with safe_open(str(run/'final_checkpoint/model.safetensors'),framework='pt') as final, safe_open(str(root/'_workspace/v2_models/ast/model.safetensors'),framework='pt') as original:
 for name in final.keys():
  value=final.get_tensor(name)
  assert torch.isfinite(value).all(),name
  if name.startswith('heads.'):
   assert not torch.equal(value,warmup_state[name]),name
   head_changes.append(name)
   continue
  previous=original.get_tensor(name.replace('backbone.','audio_spectrogram_transformer.',1))
  if name.startswith(('backbone.encoder.layer.11.','backbone.layernorm.')):
   (unchanged if torch.equal(previous,value) else changed).append(name)
  else:
   assert torch.equal(previous,value),name
   frozen+=1
assert changed
warmup=json.loads((out/'warmup_report.json').read_text())
finetune=json.loads((out/'finetune_report.json').read_text())
ids={r['ID'] for r in json.loads((out/'full_training_inventory.json').read_text())}
for report,epochs in ((warmup,8),(finetune,17)):
 assert set(report['exposures'])==ids
 assert set(report['exposures'].values())=={epochs}
 assert len(report['epochs'])==epochs
 assert all(r['unique_files']==1029 and r['input_count']==1029 for r in report['epochs'])
 assert report['independent_validation_score'] is None
last_state=torch.load(run/'finetune/last.pt',weights_only=False,map_location='cpu')
assert last_state['epoch']==17
with safe_open(str(run/'final_checkpoint/model.safetensors'),framework='pt') as exported:
 for name,value in last_state['model'].items():
  assert torch.equal(exported.get_tensor(name),value),name
(out/'final_weight_audit.json').write_text(json.dumps({'all_final_tensors_finite':True,'export_equals_epoch17_trainable_state':True,'heads_changed_since_full_warmup':head_changes,'frozen_tensors_identical_to_pretrained':frozen,'trainable_backbone_tensors_changed':changed,'trainable_backbone_tensors_unchanged':unchanged,'warmup_total_exposures':1029*8,'finetune_total_exposures':1029*17,'files_per_epoch':1029,'private_competition_evaluation_data_used':False},indent=2)+'\n')
with (out/'epoch_history.csv').open('w',newline='') as stream:
 writer=csv.DictWriter(stream,fieldnames=['stage','epoch','mean_loss','unique_files','input_count','head_gradient','backbone_gradient','absolute_parameter_update','seconds'])
 writer.writeheader()
 for report in (warmup,finetune):
  for epoch in report['epochs']:
   writer.writerow({'stage':report['stage'],**{k:epoch[k] for k in writer.fieldnames if k!='stage'}})
archive=next(run.glob('*.zip'))
shutil.copy2(archive,submission/archive.name)
assert hashlib.sha256(archive.read_bytes()).hexdigest()==hashlib.sha256((submission/archive.name).read_bytes()).hexdigest()
files=['full_refit.py','run_full_refit.py','build_full_refit_notebook.py','DACON_236749_full_refit.ipynb','train_v2.py','ast_multitask.py','v2_inference.py','graph_features.py','graph_fusion.py','graph_scoring.py','run_experiment.py','package_submission.py','package_v2_submission.py','AST_LICENSE.txt','test_full_refit.py','test_ast_multitask.py']
(out/'source_hashes.json').write_text(json.dumps({name:hashlib.sha256((root/name).read_bytes()).hexdigest() for name in files},indent=2)+'\n')
with zipfile.ZipFile(out/'reproduction_sources.zip','w',zipfile.ZIP_DEFLATED) as packed:
 for name in files: packed.write(root/name,name)
print(json.dumps({'finished':finished,'weight_audit':json.loads((out/'final_weight_audit.json').read_text()),'zip':str(submission/archive.name)},indent=2))


## Epoch 기록

아래 값은 학습 batch loss입니다. 기존 개발·평가 자료를 모두 학습에 사용했으므로 독립 자체 평가 점수는 없습니다.

In [ ]:
import csv
from pathlib import Path
with Path('results/full_refit_20260913/epoch_history.csv').open() as stream:
    rows = list(csv.DictReader(stream))
print('stage, epoch, mean_batch_loss, unique_files')
for row in rows:
    print(row['stage'],row['epoch'],row['mean_loss'],row['unique_files'],sep=', ')
print('Official score: unmeasured; submission: not sent')